# ALBench-S2F pre-flight diagnostics

Loads results from `results/preflight/<task>/...` and produces:
- D_min sweep curves (Task 2, 9)
- LR × BS heatmaps per arch (Task 3)
- Train/val curves for epoch budget calibration (Task 4)
- Augmentation deltas (Task 5)
- Parameterization sensitivity bars (Task 6)
- Dropout sensitivity bars (Task 7)
- Acquisition Jaccard distances (Task 8)

All uncertainty bars are **empirical ranges** per PI directive:
- n ≥ 4 seeds: 2nd-lowest .. 2nd-highest
- n = 3 seeds: midpoint(low, mid) .. midpoint(mid, high)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path(".").resolve().parents[0]
PREFLIGHT_DIR = REPO / "results" / "preflight"
PREFLIGHT_DIR

## Empirical range helper (PI directive)

In [ ]:
def empirical_range(vals):
    """Return (low, high) per pre-flight protocol.

    n >= 4:  range = [2nd lowest, 2nd highest]
    n == 3:  range = [(low+mid)/2, (mid+high)/2]
    n <= 2:  range = (mean, mean) — degenerate band
    """
    s = sorted(float(v) for v in vals)
    n = len(s)
    if n >= 4:
        return s[1], s[-2]
    if n == 3:
        return (s[0] + s[1]) / 2.0, (s[1] + s[2]) / 2.0
    m = float(np.mean(s)) if s else float("nan")
    return m, m

## Result loader

In [ ]:
def load_results(pattern="**/result.json"):
    rows = []
    for f in PREFLIGHT_DIR.rglob("result.json"):
        try:
            r = json.loads(f.read_text())
        except Exception as e:
            print("skip", f, e)
            continue
        r["_path"] = str(f.relative_to(REPO))
        rows.append(r)
    return pd.DataFrame(rows)


df = load_results()
df.head()